In [0]:
%sql
USE CATALOG engineering_labs;
USE SCHEMA 01_bronze;

In [0]:
import yaml
from pathlib import Path

In [0]:
notebook_path = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

notebook_dir = Path(notebook_path).parent

yaml_path = (
    Path("/Workspace")
    / notebook_dir.parent
    / "config"
    / "ingestion_metadata.yaml"
)

print(yaml_path)

In [0]:
config

In [0]:
for table in config["tables"]:
    print(f"{table['name']}")

In [0]:
def validate_metadata(table):
    print(f"Validating metadata for {table["name"]}...")
    required_fields = [
        "name",
        "source",
        "target",
        "format"
    ]

    for f in required_fields: 
        if f not in table: 
            raise ValueError(
                f"Missing required field: `{f}` in YAML."
            )
    print(f"Metadata validation successful for: {table["name"]}")

In [0]:
def create_table(target):
    print(f"Creating {target} table")
    spark.sql(f"""
              CREATE TABLE IF NOT EXISTS {target}
              USING DELTA
              """)

In [0]:
def copy_into(source, target, file_format):
    print(f"Ingesting {target}")
    spark.sql(f"""
                    COPY INTO {target}
                    FROM '{source}' 
                    FILEFORMAT = {file_format.upper()}
                    FORMAT_OPTIONS (
                        'header' = 'true'
                     -- 'inferSchema' = 'true' 
                    )
                    COPY_OPTIONS ('mergeSchema' = 'true'
                    )
                """)

In [0]:
def ingestion(): 
    for t in config["tables"]:

        validate_metadata(t)

        source = t["source"]
        target = t["target"]
        file_format = t['format']

        create_table(target)
        copy_into(source, target, file_format)
        print(f"Successfully ingested {target}")

In [0]:
ingestion()